Load dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets, Audio, get_dataset_split_names

# 1. Fetch dataset metadata to automatically extract all available configurations (languages/sub-datasets)
configs = get_dataset_split_names("amu-cai/CAMEO")
print(f"Found {len(configs)} subsets: {configs}")

# 2. Load all subsets
print("Loading and merging data... This may take a moment.")
datasets_list = [
    load_dataset("amu-cai/CAMEO", split=f'{conf}[:50%]')
    for conf in configs
]


full_dataset = concatenate_datasets(datasets_list)
print(full_dataset)

full_dataset = full_dataset.class_encode_column("emotion")
print(full_dataset[0])

sampling_rate = 16000
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=sampling_rate))
print(full_dataset[0])



In [ ]:
print(full_dataset[0])


In [ ]:
emotion_groups = {
    # Angry
    'anger':         'angry',
    'disgust':       'angry',
    'fear':          'angry',
    'anxiety':       'angry',

    # Happy
    'happiness':     'happy',
    'excitement':    'happy',
    'enthusiasm':    'happy',
    'encouragement': 'happy',

    # Neutral
    'neutral':       'neutral',
    'assertiveness': 'neutral',
    'sarcasm':       'neutral',
    'calm':          'neutral',

    # Sad
    'sadness':       'sad',
    'concern':       'sad',
    'apology':       'sad',
    'surprise':      'sad',
}

# New mappings
labels = ['angry', 'happy', 'neutral', 'sad']
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(label2id)
print(id2label)

def remap_emotion(batch):
    original_emotion_name = full_dataset.features["emotion"].int2str(batch["emotion"])
    new_emotion_name = emotion_groups[original_emotion_name]
    batch["emotion"] = label2id[new_emotion_name]
    return batch

full_dataset = full_dataset.map(remap_emotion)

print(full_dataset[0])


In [ ]:
from transformers import SeamlessM4TFeatureExtractor, AutoConfig, AutoModelForAudioClassification

model_id = "facebook/w2v-bert-2.0"

# 1. Load Feature Extractor instead of AutoProcessor
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(model_id)

config = AutoConfig.from_pretrained(
    model_id,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
    finetuning_task="audio-classification"
)

# 3. Load Model
model = AutoModelForAudioClassification.from_pretrained(
    model_id, 
    config=config,
    ignore_mismatched_sizes=True
)

In [ ]:
from datasets import config
print(config.HF_DATASETS_CACHE)

In [ ]:
# 4. Prepare Dataset Function
def prepare_dataset(batch):
    audio = batch["audio"]
    inputs = feature_extractor(
        audio["array"],
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = batch["emotion"] 
    
    return batch

# 5. Map with a smaller writer_batch_size to avoid RAM issues
encoded_dataset = full_dataset.map(
    prepare_dataset,
    remove_columns=["audio"], 
    # num_proc=4,
    writer_batch_size=100 
)



In [ ]:
import numpy as np


print(encoded_dataset[0])

# Train/validation split
final_splits = encoded_dataset.train_test_split(test_size=0.15, seed=42)

print("\nDataset summary:")
print(f"Total number of samples: {len(encoded_dataset)}")
print(f"Split: {final_splits['train']}")
print(f"Split: {final_splits['test']}")


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    feature_extractor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        if "labels" in features[0]:
            batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
            
        return batch

# Initialize the collator
data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
from transformers import TrainerCallback

class ProgressCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*50}")
        print(f"Epoch {int(state.epoch) + 1} / {args.num_train_epochs}")
        print(f"{'='*50}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        total = state.max_steps
        pct = step / total * 100

        if "loss" in logs:
            print(f"  Step {step}/{total} ({pct:.1f}%) | loss: {logs['loss']:.4f} | lr: {logs.get('learning_rate', 0):.2e}")

        if "eval_accuracy" in logs:
            print(f"  >> Evaluation | accuracy: {logs['eval_accuracy']:.4f} | eval_loss: {logs.get('eval_loss', 0):.4f}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n  Epoch {int(state.epoch)} completed.")


In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./w2v-bert-emotion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5, 
    per_device_train_batch_size=4,      
    per_device_eval_batch_size=4,       
    gradient_accumulation_steps=16,     
    num_train_epochs=20,
    logging_steps=10,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    gradient_checkpointing=True,     
    dataloader_pin_memory=False,   
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_splits['train'],
    eval_dataset=final_splits['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback()],  
)


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"Free:      {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")


In [ ]:
trainer.train()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(final_splits['test'])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

id2label = {v: k for k, v in label2id.items()}
pred_names = [id2label[p] for p in preds]
label_names = [id2label[l] for l in labels]

print(classification_report(label_names, pred_names))

In [ ]:
# 1. Run final evaluation on the test split
metrics = trainer.evaluate()
print("\n--- Final Evaluation Statistics ---")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

# 2. View per-class performance 
predictions = trainer.predict(final_splits["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

from sklearn.metrics import classification_report
report = classification_report(labels, preds, target_names=id2label.values())
print("\nDetailed Classification Report:")
print(report)

In [ ]:
# First, make sure you are logged in
# Run this in a terminal: huggingface-cli login

from huggingface_hub import notebook_login; notebook_login()

repo_name = "wav2vec2-bert-emotion-multilingual-cameo"

# Save locally first as a backup
trainer.save_model("./final_model")
feature_extractor.save_pretrained("./final_model")

# Push to Hugging Face Hub
# This will upload the weights, config, and feature extractor automatically
trainer.push_to_hub(repo_name)

print(f"Model successfully uploaded to: https://huggingface.co/dziobak/{repo_name}")